# nb_setup_03_create_search_index — define the Azure AI Search index

Idempotently creates (or updates) the vector index used for RAG. Fields include a vector
column, page numbers, and `allowed_groups` for **query-time security trimming**.

Run once, and again whenever the index schema changes. See `PRODUCT_SPEC.md` sections 7.2, 10, 13.

This notebook uses the **Azure AI Search REST API** via `requests` (always present in Fabric) so
there is **no `%pip install`** — inline pip is rejected by the Fabric job runtime.

## Required permissions (identity running this notebook)
Auth is **hybrid** (a Fabric constraint). This notebook only talks to AI Search:

| Resource | Auth in Fabric |
| --- | --- |
| Azure AI Search | **Admin API key** from Key Vault (`kv_name`/`search_key_secret`) |

The running user needs Key Vault **secret get** on the vault. AI Search must have key auth enabled
(`authOptions`). Cognitive Services (DI/AOAI) are keyless Entra — see nb_pipeline_02. The Fabric
workspace/lakehouse must be attached so `spark.table('config')` resolves.


## Config
Index settings come from the `config` delta table (seeded by `nb_setup_01_bootstrap`).


In [ ]:
cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
SEARCH_ENDPOINT = cfg['search_endpoint'].rstrip('/')
INDEX_NAME      = cfg['search_index_name']
DIMENSIONS      = int(cfg['embedding_dimensions'])
API_VERSION     = '2024-07-01'
print('endpoint:', SEARCH_ENDPOINT, '| index:', INDEX_NAME, '| dims:', DIMENSIONS)


## Auth — AI Search admin key from Key Vault
In Fabric, `DefaultAzureCredential` can't obtain tokens and the AI Search audience
(`https://search.azure.com`) is not issuable via `notebookutils.credentials.getToken`. Since AI
Search is **not** under the Cognitive Services `disableLocalAuth` policy, we authenticate with an
**admin API key read from Key Vault** (`kv_name` / `search_key_secret`) and pass it in the
`api-key` header on every REST call.


In [ ]:
import notebookutils, requests

VAULT_URL = f"https://{cfg['kv_name']}.vault.azure.net/"
SEARCH_KEY = notebookutils.credentials.getSecret(VAULT_URL, cfg['search_key_secret'])
SEARCH_HEADERS = {'api-key': SEARCH_KEY, 'Content-Type': 'application/json'}
print('search key loaded, len =', len(SEARCH_KEY))


## Define + create/update the index
A `PUT /indexes/{name}` is create-or-update (idempotent). The schema carries a vector field
(HNSW), a `page_number`, and `allowed_groups` for security trimming.


In [ ]:
index_def = {
    'name': INDEX_NAME,
    'fields': [
        {'name': 'chunk_id',       'type': 'Edm.String', 'key': True},
        {'name': 'file_path',      'type': 'Edm.String', 'filterable': True},
        {'name': 'file_name',      'type': 'Edm.String', 'filterable': True, 'sortable': True},
        {'name': 'file_extension', 'type': 'Edm.String', 'filterable': True, 'facetable': True},
        {'name': 'content',        'type': 'Edm.String', 'searchable': True},
        {'name': 'content_vector', 'type': 'Collection(Edm.Single)', 'searchable': True,
         'dimensions': DIMENSIONS, 'vectorSearchProfile': 'vprofile'},
        {'name': 'page_number',    'type': 'Edm.Int32', 'filterable': True, 'sortable': True},
        {'name': 'chunk_index',    'type': 'Edm.Int32', 'sortable': True},
        # Sprint 11 metadata fields (folder-based filtering/faceting + source attributes).
        {'name': 'folder_path',    'type': 'Edm.String', 'filterable': True, 'facetable': True, 'sortable': True},
        {'name': 'file_size',      'type': 'Edm.Int64', 'filterable': True, 'sortable': True},
        {'name': 'last_modified',  'type': 'Edm.DateTimeOffset', 'filterable': True, 'sortable': True},
        # Author from document properties (Office docProps/core.xml or PDF /Author); null when absent.
        {'name': 'author',           'type': 'Edm.String', 'filterable': True, 'facetable': True},
        {'name': 'last_modified_by', 'type': 'Edm.String', 'filterable': True, 'facetable': True},
        # Security trimming: the Entra group GUIDs allowed to see this chunk. retrievable:False
        # keeps the ACL list out of query responses (Azure AI Search best practice).
        {'name': 'allowed_groups', 'type': 'Collection(Edm.String)', 'filterable': True, 'retrievable': False},
        {'name': 'chunk_strategy_version', 'type': 'Edm.String', 'filterable': True},
        {'name': 'indexed_utc',    'type': 'Edm.DateTimeOffset', 'filterable': True, 'sortable': True},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'hnsw', 'kind': 'hnsw'}],
        'profiles':   [{'name': 'vprofile', 'algorithm': 'hnsw'}],
    },
    'semantic': {
        'configurations': [{
            'name': 'default',
            'prioritizedFields': {'prioritizedContentFields': [{'fieldName': 'content'}]},
        }],
    },
}

url = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}?api-version={API_VERSION}'
resp = requests.put(url, headers=SEARCH_HEADERS, json=index_def)
if resp.status_code in (400, 409):
    # Azure AI Search cannot remove/retype fields via an update PUT (e.g. dropping
    # 'embedding_model'). Recreate the index to apply the new schema. DESTRUCTIVE: this
    # deletes all documents; re-run the ingest pipeline (or the E2E) to repopulate.
    print(f'PUT rejected ({resp.status_code}): {resp.text[:300]}')
    print('Schema change is not updatable in place -> deleting and recreating the index...')
    d = requests.delete(url, headers=SEARCH_HEADERS)
    if d.status_code not in (200, 204, 404):
        raise RuntimeError(f'index delete failed {d.status_code}: {d.text}')
    resp = requests.put(url, headers=SEARCH_HEADERS, json=index_def)
if resp.status_code not in (200, 201):
    raise RuntimeError(f'index create failed {resp.status_code}: {resp.text}')
print('index ready:', resp.json()['name'])


## Query-time security trimming (reference)
The consuming app filters on the caller's Entra group membership, e.g.:

```
filter = "allowed_groups/any(g: search.in(g, '<comma-separated-user-group-guids>'))"
```

How the app obtains the caller's groups (e.g. OBO) is an open question — see spec §13.
